In [ ]:
# Removendo mensagens de erro:
import warnings

warnings.filterwarnings(
    "ignore",
    message=r".*is_datetime64tz_dtype is deprecated.*",
    category=DeprecationWarning,
)

## 1. Extraindo os Dados:

#### Importando bibliotecas essenciais para a Análise de Dados:

In [ ]:
# Importando bibliotecas:
from datetime import time, timedelta, datetime              # Corrigindo os horários e as datas...
from glob import glob                                       # Busca de arquivos por padrões...
import pandas as pd                                         # Análise e manipulação de dados...
import numpy as np                                          # Tratamento de Dados...
import warnings                                             # Controle de avisos...
import os                                                   # Operações com arquivos e sistema...
import re                                                   # Mechendo no endereço dos arquivos...

pd.set_option('mode.chained_assignment', None)              # Desativa avisos de atribuição encadeada...
warnings.filterwarnings("ignore", category=FutureWarning)   # Oculta avisos de futura depreciação...

#### Consumindo os Dados dos arquivos Parquet:

In [ ]:
# Caminhos das pastas com os arquivos '.parquet':
pasta_departamentos = R"..\Base de Dados Original\parquet\Departamentos-Afastados\Departamentos.parquet"
pasta_afastados = R"..\Base de Dados Original\parquet\Departamentos-Afastados\Afastados.parquet"
pasta_mensalistas = R"..\Base de Dados Original\parquet\Mensalistas"
pasta_gptw = R"..\Base de Dados Original\parquet\GPTW\GPTW.parquet"
pasta_executivos = R"..\Base de Dados Original\parquet\Executivos"
pasta_horistas = R"..\Base de Dados Original\parquet\Horistas"
pasta_movimentacoes = R"..\Base de Dados Original\parquet\Movimentacoes\Movimentacoes_01-01-2023_09-30-2025.parquet"

# Lista todos os arquivos '.parquet' em cada pasta:
arquivos_parquet_executivos = glob(os.path.join(pasta_executivos, "*.parquet"))
arquivos_parquet_mensalistas = glob(os.path.join(pasta_mensalistas, "*.parquet"))
arquivos_parquet_horistas = glob(os.path.join(pasta_horistas, "*.parquet"))

# Carrega os arquivos e adiciona a coluna 'Cargo':
tables = [
    pd.read_parquet(caminho).assign(Cargo=cargo)
    for arquivos, cargo in zip([arquivos_parquet_executivos, arquivos_parquet_mensalistas, arquivos_parquet_horistas], ['E', 'M', 'H'])
    for caminho in arquivos
]

# Carrega o arquivo para dentro do Data Frame:
afastados = pd.read_parquet(pasta_afastados)
departamentos = pd.read_parquet(pasta_departamentos)
gptw = pd.read_parquet(pasta_gptw)
movimentacoes = pd.read_parquet(pasta_movimentacoes)

In [ ]:
# === Resumo final ===
total_linhas = sum(len(df) for df in tables)
print("\n📊 RESUMO GERAL")
print(f"Arquivos executivos: {len(arquivos_parquet_executivos)}")
print(f"Arquivos mensalistas: {len(arquivos_parquet_mensalistas)}")
print(f"Arquivos horistas: {len(arquivos_parquet_horistas)}")
print(f"Total de DataFrames lidos: {len(tables)}")
print(f"Total de linhas combinadas: {total_linhas:,}")

## 2. Tratando os Dados:

#### Definição das Funções para Transformação e Tratamento de Dados:
1. **Renomear Colunas:**  
   Função que renomeia as colunas do DataFrame com base em um dicionário de mapeamento, garantindo padronização e clareza nos nomes.
2. **Mover Colunas:**  
   Função que move as colunas do DataFrame com base em um dicionário de mapeamento, garantindo padronização e clareza nos nomes.

In [ ]:
# Função para renomear coluna:
def renomear_coluna(df, col, newname):
    df = df.rename(columns={col: newname})
    return df

# Função para mover coluna
def mover_coluna(df, col, nova_posicao):
    colunas = list(df.columns)
    colunas.insert(nova_posicao, colunas.pop(colunas.index(col)))
    return df[colunas]

#### Aplicando Funções a lista de tabelas:

In [ ]:
# Aplicando a função a todas as colunas de todas as tabelas:
for i in range(len(tables)):
    print(f"\rProcessando tabela {i+1}/{len(tables)}", end="", flush=True)
    tables[i] = renomear_coluna(tables[i], tables[i].columns[0], 'Cadastro')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[1], 'Coletor')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[2], 'Descricao Coletor')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[3], 'Nome')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[4], 'Filial')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[5], 'Data')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[6], 'Hora')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[7], 'Acesso')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[8], 'Dir')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[9], 'Tipo')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[10], 'Uso Acesso')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[11], 'Planta')
#'Cadastro', 'Nome', 'Filial', 'Data', 'Hora', 'Dir', 'Tipo', 'Acesso', 'Coletor', 'Planta', 'Descricao Coletor', 'Uso Acesso', 'Cargo'...

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'departamentos':
departamentos = renomear_coluna(departamentos, departamentos.columns[0], 'Cadastro')
departamentos = renomear_coluna(departamentos, departamentos.columns[2], 'Heranca')
departamentos = renomear_coluna(departamentos, departamentos.columns[3], 'Cargo Especifico')
departamentos = renomear_coluna(departamentos, departamentos.columns[4], 'Departamento')
departamentos = renomear_coluna(departamentos, departamentos.columns[5], 'Area')
departamentos = renomear_coluna(departamentos, departamentos.columns[6], 'Local')

# Aplicando a função de renomear o nome da coluna 'Departamento' da tabela 'afastados':
afastados = mover_coluna(afastados, 'Diretoria RH', 2)
afastados = renomear_coluna(afastados, afastados.columns[0], 'Cadastro')
afastados = renomear_coluna(afastados, afastados.columns[2], 'Heranca')
afastados = renomear_coluna(afastados, afastados.columns[3], 'Cargo Especifico')
afastados = renomear_coluna(afastados, afastados.columns[4], 'Departamento')
afastados = renomear_coluna(afastados, afastados.columns[5], 'Area')
afastados = renomear_coluna(afastados, afastados.columns[6], 'Local')

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'gptw':
gptw = renomear_coluna(gptw, gptw.columns[0], 'Área')

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'movimentacoes':
movimentacoes = renomear_coluna(movimentacoes, movimentacoes.columns[0], 'Cadastro')

#### Concatenar todas as tabelas da lista em um único Data Frame:

In [ ]:
# Concatenando todas as tabelas em um único DataFrame:
df_ronda = pd.concat(tables, ignore_index=True)

# Ordenando as colunas na ordem desejada:
df_ronda = df_ronda[['Cadastro', 'Nome', 'Filial', 'Data', 'Hora', 'Dir', 'Tipo', 'Acesso', 'Planta', 'Coletor', 'Descricao Coletor', 'Uso Acesso', 'Cargo']]
movimentacoes = movimentacoes[
    ['Cadastro', 'Abreviação UO antiga', 'Desc. UO antiga', 'Desc. Cargo antigo',
    'Abreviação UO', 'Desc. UO', 'Desc. Cargo', 'Tipo da Movimentação', 'Data de alteração']
]
df_ronda = df_ronda.drop(df_ronda.index[-1]) # Remover última linha de total...

#### Diferenciando as pessoas que foram demitidas ou que não serão analisadas e inserindo todos os departamentos delas:

In [ ]:
# Normalização mínima do 'Cadastro' (evita 123, "123", 123.0 divergirem)
def _norm_id(s):
    s = s.astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    return s.replace({'nan': np.nan})  # volta NaN onde era NaN

for _df in (df_ronda, afastados, departamentos):
    if 'Cadastro' not in _df.columns:
        raise KeyError("Faltou a coluna 'Cadastro' em um dos DataFrames.")
    _df['Cadastro'] = _norm_id(_df['Cadastro'])

# Afastado por Cadastro:
df_ronda['Afastado'] = df_ronda['Cadastro'].isin(afastados['Cadastro']).map({True: 'Sim', False: 'Não'})

# Preparar 'departamentos' com nomes únicos de coluna:
deps = departamentos[['Cadastro','Nome completo','Area','Departamento','Heranca','Cargo Especifico']].rename(
    columns={
        'Nome completo':'Nome_dep',
        'Area':'Area_dep',
        'Departamento':'Departamento_dep',
        'Heranca':'Heranca_dep',
        'Cargo Especifico':'Cargo Especifico_dep'
    }
)

# Merge (left) pelo Cadastro:
df_ronda = df_ronda.merge(deps, on='Cadastro', how='left')

# Dicionários (afastados) por Cadastro:
map_afastados_nome = dict(zip(afastados['Cadastro'], afastados['Nome completo']))
map_afastados_area = dict(zip(afastados['Cadastro'], afastados['Area']))
map_afastados_dept = dict(zip(afastados['Cadastro'], afastados['Departamento']))
map_afastados_her  = dict(zip(afastados['Cadastro'], afastados['Heranca']))
map_afastados_carg = dict(zip(afastados['Cadastro'], afastados['Cargo Especifico']))

# Substituir com prioridade: afastados > departamentos > original:

# Nome:
df_ronda['Nome'] = (
    df_ronda['Cadastro'].map(map_afastados_nome)   # 1º: afastados…
    .fillna(df_ronda['Nome_dep'])                  # 2º: departamentos…
    .fillna(df_ronda.get('Nome'))                  # 3º: valor original (se existir)…
)

# Area:
df_ronda['Area'] = (
    df_ronda['Cadastro'].map(map_afastados_area)
    .fillna(df_ronda['Area_dep'])
)

# Departamento:
df_ronda['Departamento'] = (
    df_ronda['Cadastro'].map(map_afastados_dept)
    .fillna(df_ronda['Departamento_dep'])
)

# Heranca:
df_ronda['Heranca'] = (
    df_ronda['Cadastro'].map(map_afastados_her)
    .fillna(df_ronda['Heranca_dep'])
)

# Cargo Especifico:
df_ronda['Cargo Especifico'] = (
    df_ronda['Cadastro'].map(map_afastados_carg)
    .fillna(df_ronda['Cargo Especifico_dep'])
)

# (Opcional) Manter as colunas *_dep para auditoria; se quiser remover, descomente abaixo:
df_ronda = df_ronda.drop(columns=['Nome_dep','Area_dep','Departamento_dep','Heranca_dep','Cargo Especifico_dep'], errors='ignore')

#### Corrigindo as colunas de Datas:

In [ ]:
# Garante que seja timedelta:
df_ronda['Hora'] = pd.to_numeric(df_ronda['Hora'], errors='coerce')  # garante números
df_ronda['Hora'] = pd.to_timedelta(df_ronda['Hora'], unit='m')       # converte minutos → timedelta
df_ronda['Hora'] = (pd.to_datetime('00:00', format='%H:%M') + df_ronda['Hora']).dt.time

# Garante que seja datetime:
df_ronda['Data'] = pd.to_datetime(df_ronda['Data'])

# Formata para 'dd/mm/yyyy':
df_ronda['Data'] = df_ronda['Data'].dt.strftime('%d/%m/%Y')

# Garantir datetime nos dados de admissão/demissão:
afastados['Data_Adm'] = pd.to_datetime(afastados['Data_Adm'], dayfirst=True)
afastados['Data_Dem'] = pd.to_datetime(afastados['Data_Dem'], dayfirst=True)
departamentos['Data_Admissão'] = pd.to_datetime(departamentos['Data_Admissão'], format='%d.%m.%Y')

#### Corrigindo o tipo de dado de cada coluna do Data Frame: Melhorado pelo Nicholas Birochi

In [ ]:
# Formatar categóricos numéricos, como interger, para ficar mais bonito visualmente e formatando todas as outras colunas de dentro de 'df_ronda':
df_ronda['Cadastro'] = df_ronda['Cadastro'].astype(str)
df_ronda['Nome'] = df_ronda['Nome'].astype(str)
df_ronda['Filial'] = df_ronda['Filial'].astype(str)
df_ronda['Dir'] = df_ronda['Dir'].astype(str)
df_ronda['Tipo'] = df_ronda['Tipo'].astype(int)
df_ronda['Acesso'] = df_ronda['Acesso'].astype(str)
df_ronda['Planta'] = df_ronda['Planta'].astype(str)
df_ronda['Coletor'] = df_ronda['Coletor'].astype(str)
df_ronda['Descricao Coletor'] = df_ronda['Descricao Coletor'].astype(str)
df_ronda['Uso Acesso'] = df_ronda['Uso Acesso'].astype(int)
df_ronda['Cargo'] = df_ronda['Cargo'].astype(str)
df_ronda['Afastado'] = df_ronda['Afastado'].astype(str)
df_ronda['Area'] = df_ronda['Area'].astype(str)
df_ronda['Departamento'] = df_ronda['Departamento'].astype(str)
df_ronda['Heranca'] = df_ronda['Heranca'].astype(str)
df_ronda['Cargo Especifico'] = df_ronda['Cargo Especifico'].astype(str)

# Corrigindo os tipos das colunas de 'departamentos':
departamentos['Cadastro'] = departamentos['Cadastro'].astype(str)

#### Validando os horários de almoços dentro da fábrica:

In [ ]:
# Filtra apenas o horário de almoço:
df_almoco_sem_horario = df_ronda

# Corrigindo o formato da coluna 'Data':
df_almoco_sem_horario['Data'] = pd.to_datetime(df_almoco_sem_horario['Data'], dayfirst=True).dt.strftime('%Y-%m-%d')

# Condições de entrada e saída:
cond_ref = (
    (df_almoco_sem_horario['Dir'] == 'E') &
    (df_almoco_sem_horario['Descricao Coletor'].str.lower().str.contains('ref', na=False)) &
    (~df_almoco_sem_horario['Descricao Coletor'].str.lower().str.contains(r'\bgrill?\w*', regex=True, na=False))
)

#### Validando os horários de almoços fora da fábrica:

In [ ]:
# Aplica o filtro de horário diretamente:
df_almoco_com_horario = df_ronda[
    (df_ronda['Hora'] >= time(11, 0, 0)) &
    (df_ronda['Hora'] <= time(14, 0, 0))
]

# Cria as condições de direção e descrição:
cond_port = ((df_almoco_com_horario['Dir'] == 'S') & (df_almoco_com_horario['Descricao Coletor'].str.lower().str.contains('port')))

#### Terminando de corrigir os dados:

In [ ]:
# Aplica o filtro combinado diretamente no mesmo DataFrame:
df_almoco = df_ronda[cond_ref | cond_port]

# Removendo as duplicatas:
df_almoco = df_almoco.drop_duplicates()

#### Corrigindo o formato das colunas de Data:

In [ ]:
# Aplica o 'pd.to_datetime' para todos os DataFrames:
df_almoco_com_horario['Data'] = pd.to_datetime(df_almoco_com_horario['Data'], errors='coerce')
df_almoco_sem_horario['Data'] = pd.to_datetime(df_almoco_sem_horario['Data'], errors='coerce')
df_almoco['Data'] = pd.to_datetime(df_almoco['Data'], errors='coerce')
df_ronda['Data'] = pd.to_datetime(df_ronda['Data'], errors='coerce')

#### Base Inteira para análises específicas:

In [ ]:
# Salvando base em format '.parquet' e deixa fácil de mexer:
df_ronda.to_parquet(R"..\project\database\BaseGeral.parquet")
print("✅Carregado: '../project/database/BaseGeral.parquet'")

#### Adicionando a coluna 'Dia Semana' e corrigindo a ordem das colunas:

In [ ]:
# Inserindo os dias da semana, assim garante que a coluna 'Data' seja datetime:
df_ronda['Data'] = pd.to_datetime(df_ronda['Data'], dayfirst=True)

# Cria a coluna com o nome do dia da semana (em inglês):
df_ronda['Dia Semana'] = df_ronda['Data'].dt.strftime('%A')

# Definir base para realização dos cálculos de presença na fábrica:
df_ronda = df_ronda[['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca',
                     'Cargo Especifico', 'Dir', 'Tipo', 'Acesso', 'Planta',
                     'Coletor', 'Descricao Coletor', 'Uso Acesso', 'Cargo',
                     'Afastado', 'Dia Semana', 'Data', 'Hora']]

#### Validando dias presenciais e corrigindo colunas:

In [ ]:
# Cópia e saneamento iniciais
df_presencial = df_ronda.copy()

# --- garantir tipos/normalização ---
# Planta como string e sem espaços
df_presencial['Planta'] = df_presencial['Planta'].astype(str).str.strip()

# Sai '999' (cobre ' 999', 999, etc.)
df_presencial = df_presencial[df_presencial['Planta'] != '999']

# Data só data (00:00)
df_presencial['Data'] = pd.to_datetime(df_presencial['Data'], errors='coerce').dt.normalize()

# Hora em datetime no MESMO DIA que 'Data' (evita comparar datetime vs time)
hora_dt = pd.to_datetime(df_presencial['Hora'], errors='coerce')
df_presencial['Hora'] = df_presencial['Data'] + (hora_dt.dt.hour.astype('Int64').fillna(0).astype(int)
                                                 .astype('timedelta64[h]')) \
                        + (hora_dt.dt.minute.astype('Int64').fillna(0).astype(int)
                           .astype('timedelta64[m]') % pd.Timedelta(hours=1)) \
                        + (hora_dt.dt.second.astype('Int64').fillna(0).astype(int)
                           .astype('timedelta64[s]') % pd.Timedelta(minutes=1))

# Chaves (reduza se alguma coluna for problemática; NaN pode atrapalhar merges)
chaves = ['Cadastro','Nome','Area','Departamento','Heranca','Cargo Especifico','Afastado','Data']

# Agrega listas de plantas por pessoa/dia
plantas_por_dia = (
    df_presencial
      .groupby(chaves, dropna=False)['Planta']
      .apply(lambda s: sorted(pd.unique(s.astype(str))))
      .reset_index(name='Planta')   # já sai com coluna 'Planta' como lista
)

# Primeiros/últimos horários por pessoa/dia (usando idxmin/idxmax = mais robusto)
# observação: se houver empates (mesma Hora em várias linhas), pegamos a 1ª ocorrência
gb = df_presencial.groupby(chaves, dropna=False)['Hora']
idx_min = gb.idxmin()
idx_max = gb.idxmax()

primeiros = df_presencial.loc[idx_min, chaves + ['Hora']].rename(columns={'Hora':'Hora_primeiro'})
ultimos   = df_presencial.loc[idx_max, chaves + ['Hora']].rename(columns={'Hora':'Hora_ultimo'})

# Junta primeiro/último e plantas
df_presencial_filtrado = (
    primeiros
    .merge(ultimos, on=chaves, how='outer')
    .merge(plantas_por_dia, on=chaves, how='left')
)

# Se você QUER manter apenas as linhas do ponto do 1º e do último horário:
# fazemos um left merge de volta nas linhas originais para trazer as colunas e,
# em seguida, filtramos por Hora == primeiro/último. Isso evita o “não bateu” por tipo.
df_presencial_filtrado = (
    df_presencial.drop(columns=['Planta'], errors='ignore')
                 .merge(df_presencial_filtrado, on=chaves, how='inner')
                 .query('Hora == Hora_primeiro or Hora == Hora_ultimo')
                 .drop(columns=['Hora_primeiro','Hora_ultimo'])
                 .reset_index(drop=True)
)

# Criar um df com 'Data Real' distinta de cada pessoa - removendo duplicatas para obter a quantidade distinta de dias de presença na planta de cada pessoa somente para o presencial:
df_datas = df_presencial_filtrado.drop_duplicates(subset = ['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Afastado', 'Data'])

#### Fazendo uma análise da presença semanal de cada pessoa:

In [ ]:
# Fazer um count de 'Dia Semana' para compreender maior porcentagem de presença por dias:
df_count_por_semana = df_datas.groupby(['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Afastado'])['Dia Semana'].value_counts().reset_index()
df_count_por_semana.columns = ['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Afastado', 'Dia Semana', 'Contagem Presença']

#### Inserindo a coluna para visualizar as datas de maneira mais dinâmica:

In [ ]:
# Garante que a coluna 'Data' esteja em formato datetime:
df_datas['Data'] = pd.to_datetime(df_datas['Data'], dayfirst=True)

# Agora pode usar .dt normalmente:
df_datas['Semana por Ano'] = df_datas['Data'].dt.isocalendar().week

#### Análise dos dias presenciais por semana:

In [ ]:
# Criando a coluna 'Ano':
df_datas['Ano'] = df_datas['Data'].dt.year

# Definir base para realização dos cálculos de presença na fábrica:
df_datas = df_datas[['Nome', 'Cadastro', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Dir', 'Tipo', 'Acesso', 'Planta', 'Coletor', 'Descricao Coletor', 'Uso Acesso', 'Cargo',
                     'Afastado', 'Hora', 'Data', 'Dia Semana', 'Semana por Ano', 'Ano']]

#### Fazendo a contagem total da presença por dias:

In [ ]:
# Fazer um count de 'Semana Ano' para compreender maior porcentagem de presença por dias:
df_count_semana = df_datas[['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Cargo', 'Afastado', 'Ano', 'Semana por Ano']].value_counts().reset_index()
df_count_semana.columns = ['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Cargo', 'Afastado', 'Ano', 'Semana por Ano', 'Contagem Presença']
df_count_semana = df_count_semana.sort_values(by=['Ano', 'Semana por Ano']).reset_index(drop=True)

#### Cria uma tabela somente para o filtro das datas (Onde é definido o período aplicado para a base 'df_semana_merged'):

In [ ]:
# Calculando os dais úteis entre o período de Janeiro de 2024 até Março de 2025:
# Datas de início e fim do período:
inicio = '2024-01-01'
fim = '2025-12-31'

# Gera a lista de datas com frequência semanal (toda segunda-feira):
datas_semanais = pd.date_range(start = inicio, end = fim, freq = 'W-MON')

# Cria a tabela com início e fim de cada semana:
tabela_semanas = pd.DataFrame({
    'Início da Semana': datas_semanais,
    'Fim da Semana': datas_semanais + timedelta(days=6)
})

# Adiciona a coluna com o número da semana (ISO):
tabela_semanas['Ano'] = tabela_semanas['Início da Semana'].dt.isocalendar().year
tabela_semanas['Semana por Ano'] = tabela_semanas['Início da Semana'].dt.isocalendar().week

# Faz o merge com base em 'Cadastro', 'Ano' e 'Número da Semana':
df_semana_merged = pd.merge(
    tabela_semanas,
    df_count_semana,
    on=['Ano', 'Semana por Ano'],
    how='left'  # Mantém todas as semanas da tabela_semanas...
)

# Preenche valores ausentes com zero:
df_semana_merged['Contagem Presença'] = df_semana_merged['Contagem Presença'].fillna(0).astype(int)
df_semana_merged = df_semana_merged.sort_values(by='Contagem Presença', ascending=False)

#### Definir períodos de férias a serem removidos:

In [ ]:
# Datas das férias coletivas:
ferias = [
    (datetime(2023, 12, 18), datetime(2024, 1, 7)),
    (datetime(2024, 12, 23), datetime(2025, 1, 12)),
]

# Função para checar se uma semana está em férias:
def esta_em_ferias(inicio_semana):
    for inicio, fim in ferias:
        if inicio <= inicio_semana <= fim:
            return True
    return False

# Garantir que as datas sejam datetime:
df_semana_merged['Início da Semana'] = pd.to_datetime(df_semana_merged['Início da Semana'])
df_semana_merged['Fim da Semana'] = pd.to_datetime(df_semana_merged['Fim da Semana'])

# Filtra fora das férias:
df_filtrado = df_semana_merged[
    ~df_semana_merged['Início da Semana'].apply(esta_em_ferias)
]

# Criar lista de todas as segundas-feiras entre min e max:
data_min = df_filtrado['Início da Semana'].min()
data_max = df_filtrado['Início da Semana'].max()
todas_semanas = pd.date_range(start=data_min, end=data_max, freq='W-MON')

# Remover semanas em férias e semana 1 (se desejar):
todas_semanas = [
    df for df in todas_semanas
    if not esta_em_ferias(df) and df.isocalendar()[1] != 1
]

# Criar DataFrame de semanas com Ano e Semana por Ano:
semanas_df = pd.DataFrame({
    'Início da Semana': todas_semanas
})
semanas_df['Fim da Semana'] = semanas_df['Início da Semana'] + pd.to_timedelta(6, unit='d')
semanas_df['Ano'] = semanas_df['Início da Semana'].dt.isocalendar().year
semanas_df['Semana por Ano'] = semanas_df['Início da Semana'].dt.isocalendar().week

# Lista única de pessoas e nomes:
pessoas_df = df_filtrado[['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Cargo', 'Afastado']].drop_duplicates()

# Produto cartesiano: todas pessoas × todas semanas:
combinacoes = pd.merge(
    pessoas_df.assign(key=1),
    semanas_df.assign(key=1),
    on='key'
).drop('key', axis=1)

# Merge com dados reais:
df_semana_merged = pd.merge(
    combinacoes,
    df_filtrado[['Cadastro', 'Início da Semana', 'Contagem Presença']],
    on=['Cadastro', 'Início da Semana'],
    how='left'
)

# Preencher ausências com zero:
df_semana_merged['Contagem Presença'] = df_semana_merged['Contagem Presença'].fillna(0).astype(int)

# Ordenar cronologicamente:
df_semana_merged = df_semana_merged.sort_values(['Cadastro', 'Início da Semana']).reset_index(drop=True)

# (opcional) remover duplicados para evitar erro no map:
afastados_dedup = afastados.drop_duplicates(subset='Cadastro', keep='first')
departamentos_dedup = departamentos.drop_duplicates(subset='Cadastro', keep='first')

# Remover registros da última semana:
ultima_semana = semanas_df['Início da Semana'].max()
df_semana_merged = df_semana_merged[df_semana_merged['Início da Semana'] < ultima_semana]

# Mapear datas de afastados e departamentos com índice único
df_semana_merged['Data_Adm'] = df_semana_merged['Cadastro'].map(
    afastados_dedup.set_index('Cadastro')['Data_Adm']
)
df_semana_merged['Data_Dem'] = df_semana_merged['Cadastro'].map(
    afastados_dedup.set_index('Cadastro')['Data_Dem']
)
df_semana_merged['Data_Admissão'] = df_semana_merged['Cadastro'].map(
    departamentos_dedup.set_index('Cadastro')['Data_Admissão']
)

# Filtrar registros dentro do período válido:
df_semana_merged = df_semana_merged[
    (
        df_semana_merged['Data_Adm'].notna() &
        (df_semana_merged['Início da Semana'] >= df_semana_merged['Data_Adm']) &
        (
            df_semana_merged['Data_Dem'].isna() |
            (df_semana_merged['Início da Semana'] <= df_semana_merged['Data_Dem'])
        )
    )
    |
    (
        df_semana_merged['Data_Adm'].isna() &
        df_semana_merged['Data_Admissão'].notna() &
        (df_semana_merged['Início da Semana'] >= df_semana_merged['Data_Admissão'])
    )
]

# Remover colunas temporárias:
df_semana_merged = df_semana_merged.drop(columns=['Data_Adm', 'Data_Dem', 'Data_Admissão'])

#### Semanas com somente 'um' ou 'zero' de registro de dia presencial:

In [ ]:
# Filtra o DataFrame para manter apenas os registros com 'Contagem Presença' igual a 0 ou 1, correspondendo a pessoas com no máximo um registro de presença no dia e reseta o índice para manter a consistência após o filtro:
df_semana_merged_dia = df_semana_merged[(df_semana_merged['Contagem Presença'] == 1) | (df_semana_merged['Contagem Presença'] == 0)].reset_index(drop=True)

# Remove semana 1 por ser coletivas:
df_semana_merged_dia = df_semana_merged_dia[df_semana_merged_dia['Semana por Ano'] != 1].reset_index(drop=True)

#### Requisição por chapa:

In [ ]:
# Fazer um count de 'Semana Ano' para compreender maior porcentagem de presença por dias:
df_count_semana = df_datas[['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Cargo', 'Afastado', 'Semana por Ano', 'Ano']].value_counts().reset_index()
df_count_semana.columns = ['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Cargo', 'Afastado', 'Semana por Ano', 'Ano', 'Contagem Presença']
df_count_semana = df_count_semana.sort_values(by=['Ano', 'Semana por Ano']).reset_index(drop=True)

#### Exibindo a presença por pessoa no horário de serviço:

In [ ]:
# Merge com base em 'Ano', 'Semana por Ano', 'Cadastro' e 'Nome':
df_presenca_merged_por_cadastro = df_semana_merged

# Preenche valores ausentes com zero:
df_presenca_merged_por_cadastro['Contagem Presença'] = df_presenca_merged_por_cadastro['Contagem Presença'].fillna(0).astype(int)

#### Funções para criar a tabela corrigida de Headcount:

In [ ]:
# ------------------------------------------------------------
# Headcount & Turnover por área/ano com roll-up condicionado
# Regra:
#   HC → foto em 31/12 de cada ano (FULL + pais/avós na mesma Heranca).
#   E  → pessoas que entraram na área no ano (admissão + transferência).
#   S  → pessoas que saíram da área no ano (demissão + transferência).
#
# Transferências entre subáreas da MESMA área macro
# (ex.: E-IT ↔ E-IE) e MESMA Heranca:
#   - contam em E/S das subáreas;
#   - NÃO sobem para a área macro (linha geral), exceto quando a
#     própria área macro é origem ou destino (pai <-> filho).
# ------------------------------------------------------------

_norm = lambda x: ("" if pd.isna(x) else str(x).strip().upper())


def _hc_snapshot_from_departamentos(departamentos: pd.DataFrame, areaher: dict) -> pd.Series:
    """
    Conta HC pelo snapshot atual de 'departamentos' (vivos) e aplica roll-up por Heranca.
    Usado como HC do último ano (ancoragem).
    """
    dep = _rename_heranca(departamentos).copy()
    if dep.empty or "Area" not in dep.columns or "Cadastro" not in dep.columns:
        return pd.Series(dtype=int)

    dep["Cadastro"] = dep["Cadastro"].astype(str)
    dep["Area"] = dep["Area"].map(_norm)
    dep = dep[dep["Area"] != ""].copy()

    if "Data_Admissão" in dep.columns:
        dep["Data_Admissão"] = _to_dt(dep["Data_Admissão"])
        dep = dep.sort_values(["Cadastro", "Data_Admissão"]).drop_duplicates("Cadastro", keep="last")
    else:
        dep = dep.drop_duplicates("Cadastro", keep="last")

    full_counts = dep["Area"].value_counts().to_dict()
    rolled = _rollup_full_counts(full_counts, areaher)
    return pd.Series(rolled, dtype=int)


def _combine_event_sets(d1: dict, d2: dict) -> dict:
    """
    Une dicionários {area: set(pessoas)} somando os conjuntos.
    """
    out = {}
    for src in (d1 or {}, d2 or {}):
        for k, v in src.items():
            out.setdefault(k, set()).update(v)
    return out


def _recalc_hc_backwards(
    hc_year: dict[int, pd.Series],
    e_sets_by_year: dict[int, dict],
    s_sets_by_year: dict[int, dict],
    anos: tuple[int, ...],
    areas: list[str],
    dep_snapshot: pd.Series | None = None,
) -> dict[int, pd.Series]:
    """
    Recalcula HC para trás (HC_{y-1} = HC_y - E_y + S_y) APENAS para as áreas
    que ainda têm gente em 'departamentos' (snapshot atual).

    Regra:
      - Áreas com HC > 0 em 'departamentos':
          -> usam recálculo backwards a partir do snapshot atual (âncora).
      - Áreas sem ninguém em 'departamentos':
          ->  HC do último ano = 0
          ->  HC dos anos anteriores permanece o calculado direto via
              _headcount_por_ano (ou seja, baseado em 'afastados' + histórico),
              sem ser "puxado" para 0 para trás.
    """
    if not anos:
        return hc_year

    # se não houver snapshot de departamentos, não recalcula nada
    if dep_snapshot is None or not isinstance(dep_snapshot, pd.Series) or dep_snapshot.empty:
        return hc_year

    anos_sorted = sorted(anos)
    last_year = anos_sorted[-1]

    # garante HC reindexado em todas as áreas
    for y in anos_sorted:
        hc_year[y] = (
            hc_year.get(y, pd.Series(dtype=int))
                   .reindex(areas)
                   .fillna(0)
                   .astype(int)
        )

    # snapshot de departamentos reindexado nas mesmas áreas
    dep_snapshot = dep_snapshot.reindex(areas).fillna(0).astype(int)
    active_mask = dep_snapshot > 0   # True = área com gente em departamentos

    # prepara estruturas de HC final (cópia dos valores atuais)
    hc_final: dict[int, pd.Series] = {
        y: hc_year[y].copy() for y in anos_sorted
    }

    # pré-calcula contagens E/S por ano/área
    e_count_by_year: dict[int, pd.Series] = {}
    s_count_by_year: dict[int, pd.Series] = {}
    for y in anos_sorted:
        e_sets = e_sets_by_year.get(y, {}) or {}
        s_sets = s_sets_by_year.get(y, {}) or {}
        e_count_by_year[y] = pd.Series(
            {a: len(e_sets.get(a, set())) for a in areas}, dtype=int
        ).reindex(areas).fillna(0).astype(int)
        s_count_by_year[y] = pd.Series(
            {a: len(s_sets.get(a, set())) for a in areas}, dtype=int
        ).reindex(areas).fillna(0).astype(int)

    # 1) Áreas SEM ninguém em departamentos -> HC do último ano = 0, anos anteriores ficam como estavam
    inactive_mask = ~active_mask

    if len(anos_sorted) >= 2:
        prev_year = anos_sorted[-2]

        hc_prev = hc_final[prev_year]
        e_last = e_count_by_year[last_year]
        s_last = s_count_by_year[last_year]

        hc_last_calc = (hc_prev + e_last - s_last).clip(lower=0).astype(int)

        # aplica SOMENTE nas áreas sem snapshot
        hc_final[last_year][inactive_mask] = hc_last_calc[inactive_mask]

    # 2) Áreas COM gente em departamentos -> recalcula tudo para trás a partir da âncora do snapshot
    for idx, area in enumerate(areas):
        if not active_mask.iloc[idx]:
            continue  # pula área "morta"

        # âncora no último ano = foto atual de departamentos
        h = int(dep_snapshot.iloc[idx])
        hc_final[last_year].iloc[idx] = h

        # anda para trás nos anos
        for i in range(len(anos_sorted) - 1, 0, -1):
            y = anos_sorted[i]
            y_prev = anos_sorted[i - 1]

            e_y = int(e_count_by_year[y].iloc[idx])
            s_y = int(s_count_by_year[y].iloc[idx])

            h_prev = h - e_y + s_y
            if h_prev < 0:
                h_prev = 0

            hc_final[y_prev].iloc[idx] = h_prev
            h = h_prev

    return hc_final


def _area_clean(area: str) -> str:
    """
    Versão 'limpa' da área:
    - maiúscula
    - REMOVE tudo que não é letra/número ( *, /, -, espaços, etc.)
    Ex:
      'E-RL'    -> 'ERL'
      'E/RLA*'  -> 'ERLA'
    """
    return re.sub(r"[^A-Z0-9]", "", _norm(area))


def _build_areaher_from_df(her_df: pd.DataFrame) -> dict:
    """
    Monta um dict {Area_normalizada: Heranca_normalizada}
    a partir de um DF com colunas ['Area','Heranca'].
    Usado em _areas_macro_e_subareas.
    """
    if not isinstance(her_df, pd.DataFrame) or her_df.empty:
        return {}
    if "Area" not in her_df.columns or "Heranca" not in her_df.columns:
        return {}
    m = {}
    for _, r in her_df[["Area", "Heranca"]].dropna(subset=["Area"]).iterrows():
        a = _norm(r["Area"])
        h = _norm(r["Heranca"])
        if a and h and h != "****":
            m[a] = h
    return m


def _is_parent_of(parent: str, child: str, areaher: dict) -> bool:
    """
    True se 'parent' é ancestral de 'child' dentro da MESMA Heranca.
    Usado para descobrir subáreas de um alvo.
    """
    parent_n = _norm(parent)
    child_n = _norm(child)
    her_p = _norm(areaher.get(parent_n, ""))
    her_c = _norm(areaher.get(child_n, ""))
    if not her_p or her_p != her_c:
        return False
    pc = _area_clean(parent_n)
    cc = _area_clean(child_n)
    return len(pc) < len(cc) and cc.startswith(pc)


def _area_parents(area: str, areaher: dict) -> list[str]:
    """
    Retorna TODOS os ancestrais hierárquicos de uma área, com base em:
      - Mesma Heranca;
      - Código 'limpo' do pai é prefixo mais curto do código 'limpo' do filho.

    Ex com mesma Heranca:
      E         -> nenhum pai
      E-I       -> pais: E
      E-IT      -> pais: E-I, E
    """
    area_n = _norm(area)
    clean = _area_clean(area_n)
    her = _norm(areaher.get(area_n, ""))
    if not clean or not her:
        return []

    parents = []
    for cand, h in areaher.items():
        if _norm(h) != her:
            continue
        cand_n = _norm(cand)
        cand_clean = _area_clean(cand_n)
        if len(cand_clean) < len(clean) and clean.startswith(cand_clean):
            parents.append(cand_n)

    parents = sorted(set(parents), key=lambda a: len(_area_clean(a)), reverse=True)
    return parents


def _to_dt(s):
    return pd.to_datetime(s, errors="coerce", dayfirst=True)


def _rename_heranca(df):
    if "Herança" in df.columns and "Heranca" not in df.columns:
        df = df.rename(columns={"Herança": "Heranca"})
    return df


def _mov_id_col(df: pd.DataFrame) -> str:
    """
    Tenta descobrir a coluna de ID da pessoa nas bases de movimentação.
    Procura por nomes comuns ('Cadastro', 'Matricula', 'Nº Pessoal', etc).
    """
    if not hasattr(df, "columns"):
        raise TypeError(
            f"Objeto passado para _mov_id_col não é um DataFrame. Tipo: {type(df)}"
        )

    colunas_orig = list(df.columns)
    colunas_norm = {c: _norm(c) for c in colunas_orig}

    candidatos_exatos = {
        "CADASTRO",
        "Nº PESSOAL",
        "NO PESSOAL",
        "NUM PESSOAL",
        "ID",
        "MATRICULA",
        "MATRÍCULA",
    }

    for col, norm in colunas_norm.items():
        if norm in candidatos_exatos:
            return col

    for col, norm in colunas_norm.items():
        if any(chave in norm for chave in ["CADASTRO", "PESSOAL", "MATRIC"]):
            return col

    raise KeyError(
        "Não encontrei coluna de ID (ex.: 'Cadastro'). "
        f"Colunas disponíveis em movimentacoes: {colunas_orig}"
    )


def _consolida_bases(departamentos: pd.DataFrame, afastados: pd.DataFrame):
    """
    Monta a base consolidada de pessoas, unindo quem está em 'departamentos'
    (snapshot atual) E quem só aparece em 'afastados' (já desligados).

    Saída: DF com colunas mínimas:
      - Cadastro
      - Adm_global
      - Dem_global
      - Area_init
      - Dept_init

    Essa base é usada por _area_abrev_asof / _headcount_por_ano para montar o HC
    por ano via foto em 31/12.
    """
    # --- departamentos (snapshot atual) ---
    dep = _rename_heranca(departamentos).copy()
    if not dep.empty:
        dep["Cadastro"] = dep["Cadastro"].astype(str)
        if "Area" in dep.columns:
            dep["Area"] = dep["Area"].map(_norm)
        if "Heranca" in dep.columns:
            dep["Heranca"] = dep["Heranca"].map(_norm)
        if "Data_Admissão" in dep.columns:
            dep["Data_Admissão"] = _to_dt(dep["Data_Admissão"])
        else:
            dep["Data_Admissão"] = pd.NaT

        dep_cur = (
            dep.sort_values(["Cadastro", "Data_Admissão"])
               .drop_duplicates("Cadastro", keep="last")
        )
        dep_cur = dep_cur.rename(
            columns={
                "Heranca": "Dept_cur",
                "Area": "Area_cur",
                "Data_Admissão": "Adm_dep",
            }
        )[["Cadastro", "Dept_cur", "Area_cur", "Adm_dep"]]
    else:
        dep_cur = pd.DataFrame(columns=["Cadastro", "Dept_cur", "Area_cur", "Adm_dep"])

    # --- afastados (histórico de admissão/demissão) ---
    afa = _rename_heranca(afastados).copy()
    if not afa.empty:
        afa["Cadastro"] = afa["Cadastro"].astype(str)
        if "Area" in afa.columns:
            afa["Area"] = afa["Area"].map(_norm)
        if "Heranca" in afa.columns:
            afa["Heranca"] = afa["Heranca"].map(_norm)
        if "Data_Adm" in afa.columns:
            afa["Data_Adm"] = _to_dt(afa["Data_Adm"])
        else:
            afa["Data_Adm"] = pd.NaT
        if "Data_Dem" in afa.columns:
            afa["Data_Dem"] = _to_dt(afa["Data_Dem"])
        else:
            afa["Data_Dem"] = pd.NaT

        # primeira admissão na empresa
        adm_first = (
            afa.sort_values(["Cadastro", "Data_Adm"])
               .dropna(subset=["Data_Adm"])
               .drop_duplicates("Cadastro", keep="first")
        )
        adm_first = adm_first.rename(
            columns={
                "Heranca": "Dept_on_adm",
                "Area": "Area_on_adm",
                "Data_Adm": "Date_adm_event_first",
            }
        )[["Cadastro", "Dept_on_adm", "Area_on_adm", "Date_adm_event_first"]]

        # última demissão na empresa
        dem_last = (
            afa.sort_values(["Cadastro", "Data_Dem"])
               .dropna(subset=["Data_Dem"])
               .drop_duplicates("Cadastro", keep="last")
        )
        dem_last = dem_last.rename(
            columns={
                "Heranca": "Dept_on_dem",
                "Area": "Area_on_dem",
                "Data_Dem": "Date_dem_event_last",
            }
        )[["Cadastro", "Dept_on_dem", "Area_on_dem", "Date_dem_event_last"]]
    else:
        adm_first = pd.DataFrame(
            columns=["Cadastro", "Dept_on_adm", "Area_on_adm", "Date_adm_event_first"]
        )
        dem_last = pd.DataFrame(
            columns=["Cadastro", "Dept_on_dem", "Area_on_dem", "Date_dem_event_last"]
        )

    # --- base consolidada = união (outer) de dep_cur + afastados ---
    base = (
        dep_cur.merge(adm_first, on="Cadastro", how="outer")
               .merge(dem_last, on="Cadastro", how="outer")
    )

    # datas globais de admissão/demissão
    base["Adm_global"] = base[["Adm_dep", "Date_adm_event_first"]].min(axis=1)
    base["Dem_global"] = base["Date_dem_event_last"]

    # área/herança "inicial" (na admissão) com fallback para área atual
    base["Area_init"] = base["Area_on_adm"].where(
        base["Area_on_adm"].notna() & (base["Area_on_adm"] != ""),
        base["Area_cur"],
    )
    base["Dept_init"] = base["Dept_on_adm"].where(
        base["Dept_on_adm"].notna() & (base["Dept_on_adm"] != ""),
        base["Dept_cur"],
    )

    return base


def _area_abrev_asof(ano_dt, base, movimentacoes):
    # 1) Filtra quem estava ATIVO na data (mesma lógica de antes)
    asof = base[["Cadastro", "Area_init", "Adm_global", "Dem_global"]]
    asof = asof[
        (asof["Adm_global"].notna())
        & (asof["Adm_global"] <= ano_dt)
        & (asof["Dem_global"].isna() | (asof["Dem_global"] > ano_dt))
    ].copy()

    mv = movimentacoes.copy()
    idcol = _mov_id_col(mv)
    mv[idcol] = mv[idcol].astype(str)

    # Normaliza
    mv["Abreviação UO"] = mv["Abreviação UO"].map(_norm)
    mv["Abreviação UO antiga"] = mv["Abreviação UO antiga"].map(_norm)
    mv["Data de alteração"] = _to_dt(mv["Data de alteração"])

    # Só linhas válidas (alguma movimentação real)
    mv = mv[
        mv["Data de alteração"].notna()
        & (mv["Abreviação UO"] != "")
        & (mv["Abreviação UO antiga"] != "")
    ].sort_values([idcol, "Data de alteração"])

    # 2) Para cada pessoa, pega:
    #    - a PRIMEIRA área antiga (antes de qualquer movimentação) -> "origem histórica"
    earliest_old = (
        mv.groupby(idcol)["Abreviação UO antiga"]
          .first()
          .rename("Area_earliest_old")
    )

    #    - a ÚLTIMA área nova até a data de corte (ano_dt) -> "posição vigente"
    mv_upto = mv[mv["Data de alteração"] <= ano_dt]
    last_new = (
        mv_upto.groupby(idcol)["Abreviação UO"]
               .last()
               .rename("Area_last_upto")
    )

    # 3) Junta tudo com a base "asof"
    eff = asof.merge(earliest_old, left_on="Cadastro", right_index=True, how="left")
    eff = eff.merge(last_new, left_on="Cadastro", right_index=True, how="left")

    # 4) Regra final da área "as of":
    #    - se tem movimentação até a data → usa última área nova
    #    - senão, se já tem alguma movimentação futura → usa primeira área antiga
    #    - senão → cai no Area_init (como antes)
    eff["Area_asof"] = eff["Area_last_upto"].where(
        eff["Area_last_upto"].notna() & (eff["Area_last_upto"] != ""),
        eff["Area_earliest_old"].where(
            eff["Area_earliest_old"].notna() & (eff["Area_earliest_old"] != ""),
            eff["Area_init"],
        ),
    )

    return eff.set_index("Cadastro")["Area_asof"]


def _accumulate_events(df, id_col, area_col, her_col, areaher, bucket: dict):
    """
    Atualiza bucket[area] -> set(pids) em:
      - FULL (área exata)
      - + todos os pais/avós na MESMA Heranca.

    Usado para:
      - Entradas externas (admissão)
      - Saídas externas (demissão)
    """
    for _, r in df.iterrows():
        pid = str(r[id_col])
        area_raw = r[area_col]
        if not area_raw:
            continue

        full = _norm(area_raw)
        if not full:
            continue

        # área exata
        bucket.setdefault(full, set()).add(pid)

        # pais/avós na mesma Heranca
        for parent in _area_parents(full, areaher):
            bucket.setdefault(parent, set()).add(pid)


def _headcount_por_ano(base, movimentacoes, anos, areaher):
    out = {}
    for y in anos:
        dt = pd.Timestamp(y, 12, 31)
        area_asof = (
            _area_abrev_asof(dt, base, movimentacoes)
            .dropna()
            .astype(str)
            .map(_norm)
        )
        full_counts = area_asof.value_counts().to_dict()
        out[y] = pd.Series(_rollup_full_counts(full_counts, areaher), dtype=int)
    return out  # {ano: Series}


def _entradas_por_ano(
    afastados: pd.DataFrame,
    departamentos: pd.DataFrame,
    anos,
    areaher,
    movimentacoes: pd.DataFrame | None = None,
):
    """
    Entradas por ano (admissão na empresa), com ajuste para o caso em que:

      - A pessoa está em 'departamentos' numa área que só foi alcançada
        depois de uma movimentação lateral em 'movimentacoes'.

    Regras:
      - Usa 'afastados' como fonte principal de Data_Adm + Área.
      - Usa 'departamentos' apenas para quem não aparece em 'afastados'.
      - Para esses casos de 'departamentos':
          * se houver movimentação registrada:
              → área de entrada = primeira 'Abreviação UO antiga' (área original),
                desde que a Data_Admissão seja <= à data dessa primeira mov.
          * se não houver movimentação:
              → área de entrada = 'Area' do próprios 'departamentos'.
    """
    # --- normaliza bases de admissão externa ---
    afa = _rename_heranca(afastados).copy()
    dep = _rename_heranca(departamentos).copy()

    for df, dcol in ((afa, "Data_Adm"), (dep, "Data_Admissão")):
        if df.empty:
            continue
        df["Cadastro"] = df["Cadastro"].astype(str)
        if "Area" in df.columns:
            df["Area"] = df["Area"].map(_norm)
        if "Heranca" in df.columns:
            df["Heranca"] = df["Heranca"].map(_norm)
        df[dcol] = _to_dt(df[dcol])

    # --- mapa opcional: para cada pessoa, (primeira_area_antiga, data_primeira_mov) ---
    orig_info = {}
    if isinstance(movimentacoes, pd.DataFrame) and not movimentacoes.empty:
        mv = movimentacoes.copy()
        idcol = _mov_id_col(mv)
        mv[idcol] = mv[idcol].astype(str)

        mv["Abreviação UO antiga"] = mv["Abreviação UO antiga"].map(_norm)
        mv["Data de alteração"] = _to_dt(mv["Data de alteração"])

        mv = mv[
            mv["Data de alteração"].notna()
            & (mv["Abreviação UO antiga"] != "")
        ].sort_values([idcol, "Data de alteração"])

        # pega, para cada pessoa, a PRIMEIRA linha de movimentação
        first_mv = mv.groupby(idcol).first().reset_index()
        orig_info = {
            str(r[idcol]): {
                "area_old": r["Abreviação UO antiga"],
                "dt_first_mv": r["Data de alteração"],
            }
            for _, r in first_mv.iterrows()
        }

    def _area_entrada_dep(row):
        """
        Decide qual área considerar na ENTRADA (admissao) para quem
        só está em 'departamentos'.
        """
        cad = str(row["Cadastro"])
        info = orig_info.get(cad)
        if not info:
            # nunca teve movimentação -> considera área atual do departamentos
            return row["Area"]

        area_old = info["area_old"]
        dt_first_mv = info["dt_first_mv"]
        dt_adm = row.get("Data_Admissão")

        # se não tivermos datas consistentes, não arriscamos mexer
        if pd.isna(dt_adm) or pd.isna(dt_first_mv):
            return row["Area"]

        # caso normal: a pessoa foi admitida antes da primeira mov.
        # então, na admissão, ela estava na área_old (não na área atual de departamentos).
        if dt_adm <= dt_first_mv:
            return area_old

        # se a primeira movimentação for ANTES da Data_Admissão,
        # significa que essa Data_Admissão é de um novo vínculo.
        # Nesse cenário, faz mais sentido manter a área atual mesmo.
        return row["Area"]

    out = {}
    for y in anos:
        sets = {}

        # 1) Admissões vindas de 'afastados' no ano Y
        ev_afa = afa[afa["Data_Adm"].dt.year.eq(y)].copy()
        if not ev_afa.empty:
            for _, r in ev_afa.iterrows():
                pid = str(r["Cadastro"])
                area = _norm(r["Area"])
                if not area:
                    continue
                # >>> só conta na área exata, NÃO sobe para os pais
                sets.setdefault(area, set()).add(pid)


        pids_afa = set(ev_afa["Cadastro"]) if not ev_afa.empty else set()

        # 2) Admissões que só conseguimos ver em 'departamentos' no ano Y
        ev_dep = dep[dep["Data_Admissão"].dt.year.eq(y)].copy()
        if not ev_dep.empty:
            # tira quem já foi contado em 'afastados'
            ev_dep = ev_dep[~ev_dep["Cadastro"].isin(pids_afa)]

            if not ev_dep.empty:
                # corrige área de entrada para quem já teve movimentação lateral
                if orig_info:
                    ev_dep["Area"] = ev_dep.apply(_area_entrada_dep, axis=1)

                _accumulate_events(ev_dep, "Cadastro", "Area", "Heranca", areaher, sets)

        out[y] = sets

    return out


def _saidas_por_ano(afastados, anos, areaher, valid_ids=None):
    afa = _rename_heranca(afastados).copy()
    afa["Cadastro"] = afa["Cadastro"].astype(str)

    # Filtro: só considero demissões de quem está na base
    if valid_ids is not None:
        afa = afa[afa["Cadastro"].isin(valid_ids)]

    afa["Area"] = afa["Area"].map(_norm)
    if "Heranca" in afa.columns:
        afa["Heranca"] = afa["Heranca"].map(_norm)
    afa["Data_Dem"] = _to_dt(afa["Data_Dem"])

    out = {}
    for y in anos:
        sets = {}
        ev = afa[afa["Data_Dem"].dt.year.eq(y)].copy()
        if not ev.empty:
            _accumulate_events(ev, "Cadastro", "Area", "Heranca", areaher, sets)
        out[y] = sets
    return out


def _nearest_parent(area: str, areaher: dict) -> str:
    """
    Retorna o pai imediato (ancestral mais próximo) na MESMA Heranca.
    Se não encontrar via 'areaher', tenta um fallback por prefixo de área.
    """
    a = _norm(area)
    if not a:
        return ""

    # via mapeamento de herança
    parents = _area_parents(a, areaher)
    if parents:
        return parents[0]  # _area_parents já retorna do mais próximo para o mais distante

    # fallback por prefixo (ex.: "E-OLS-2" -> "E-OLS")
    if "-" in a:
        return a.rsplit("-", 1)[0]

    return ""


def _same_immediate_macro(a: str, b: str, areaher: dict) -> bool:
    """
    True se 'a' e 'b' têm o MESMO pai imediato e a MESMA Heranca.
    Usa _nearest_parent e compara Heranca via 'areaher'.
    """
    a_n, b_n = _norm(a), _norm(b)
    if not a_n or not b_n:
        return False

    her_a = _norm(areaher.get(a_n, ""))
    her_b = _norm(areaher.get(b_n, ""))
    if not her_a or her_a != her_b:
        return False

    pa = _nearest_parent(a_n, areaher)
    pb = _nearest_parent(b_n, areaher)
    return bool(pa) and ( _norm(pa) == _norm(pb) )


def _is_parent_child_sameher(a: str, b: str, areaher: dict) -> bool:
    """
    True se a relação for pai ↔ filho (ancestral direto ou não),
    mantendo a MESMA Heranca.
    """
    a_n, b_n = _norm(a), _norm(b)
    her_a = _norm(areaher.get(a_n, ""))
    her_b = _norm(areaher.get(b_n, ""))
    if not her_a or her_a != her_b:
        return False
    # Usa a checagem já existente
    return _is_parent_of(a_n, b_n, areaher) or _is_parent_of(b_n, a_n, areaher)


def _transfer_por_ano(movimentacoes, anos, areaher):
    """
    Transferências INTERNAS (entre áreas) por ano.

    Regras:
      - Sempre conta nas ÁREAS EXATAS:
          S[área_antiga] += 1
          E[área_nova]   += 1

      - Entre subáreas da MESMA macro (mesmo pai imediato) e MESMA Heranca,
        ou pai ↔ filho (também na mesma Heranca):
          → NÃO sobe para pai/avô (fica só nas áreas exatas).

      - Demais casos (macros diferentes OU Herancas diferentes):
          → entrada/saída de árvore:
              * pais/avós da área nova recebem E;
              * pais/avós da área antiga recebem S.
    """
    mv = movimentacoes.copy()
    idcol = _mov_id_col(mv)

    mv[idcol] = mv[idcol].astype(str)
    mv["Abreviação UO"] = mv["Abreviação UO"].map(_norm)
    mv["Abreviação UO antiga"] = mv["Abreviação UO antiga"].map(_norm)
    mv["Data de alteração"] = _to_dt(mv["Data de alteração"])

    # Só movimentações reais (data válida, áreas não vazias e diferentes)
    mv = mv[
        mv["Data de alteração"].notna()
        & (mv["Abreviação UO"] != "")
        & (mv["Abreviação UO antiga"] != "")
        & (mv["Abreviação UO"] != mv["Abreviação UO antiga"])
    ]

    ent, sai = {}, {}
    for y in anos:
        e_map, s_map = {}, {}
        ano_mv = mv[mv["Data de alteração"].dt.year.eq(y)].copy()

        for _, r in ano_mv.iterrows():
            pid = str(r[idcol])

            old_full = _norm(r["Abreviação UO antiga"])
            new_full = _norm(r["Abreviação UO"])
            if not old_full or not new_full:
                continue

            # 1) Sempre conta nas áreas EXATAS
            s_map.setdefault(old_full, set()).add(pid)
            e_map.setdefault(new_full, set()).add(pid)

            # 2) Regras de subida
            her_old = _norm(areaher.get(old_full, ""))
            her_new = _norm(areaher.get(new_full, ""))

            # Caso “mesma macro” correto:
            same_immediate = _same_immediate_macro(old_full, new_full, areaher)

            # Caso pai ↔ filho (mesma Heranca):
            parent_child = _is_parent_child_sameher(old_full, new_full, areaher)

            # Entre irmãs da MESMA macro OU pai ↔ filho → NÃO sobe.
            if same_immediate or parent_child:
                continue

            # Demais casos → sobe para pais/avós
            old_parents = _area_parents(old_full, areaher)
            new_parents = _area_parents(new_full, areaher)

            for parent_new in new_parents:
                e_map.setdefault(parent_new, set()).add(pid)
            for parent_old in old_parents:
                s_map.setdefault(parent_old, set()).add(pid)

        ent[y], sai[y] = e_map, s_map

    return ent, sai


def _sets_to_series(area_sets: dict, areas) -> pd.Series:
    return (
        pd.Series({k: len(v) for k, v in area_sets.items()}, dtype=int)
        .reindex(areas)
        .fillna(0)
        .astype(int)
    )


def _map_area_to_heranca(departamentos: pd.DataFrame, afastados: pd.DataFrame) -> dict:
    frames = []
    for df in (_rename_heranca(departamentos), _rename_heranca(afastados)):
        if {"Area", "Heranca"}.issubset(df.columns):
            t = df[["Area", "Heranca"]].copy()
            t["Area"] = t["Area"].map(_norm)
            t["Heranca"] = t["Heranca"].map(_norm)
            t = t[(t["Area"] != "") & (t["Heranca"] != "") & (t["Heranca"] != "****")]
            frames.append(t)
    if not frames:
        return {}
    cat = pd.concat(frames, ignore_index=True)
    return cat.groupby("Area")["Heranca"].agg(
        lambda s: s.mode().iat[0] if not s.mode().empty else ""
    ).to_dict()


def _areas_macro_e_subareas(area_label, piv_index, her_df=None):
    alvo = (area_label or "").strip()
    if not alvo:  # sem área → só placeholder
        return ['—']

    # monta mapa area->heranca a partir do her_df (se existir)
    if isinstance(her_df, pd.DataFrame) and {'Area', 'Heranca'}.issubset(her_df.columns):
        areaher_loc = _build_areaher_from_df(her_df)
    else:
        areaher_loc = {}

    # universo de áreas possíveis nesse pivot
    if isinstance(piv_index, (list, pd.Index)):
        all_areas = [str(a).strip() for a in list(piv_index)]
    else:
        all_areas = []

    alvo_n = _norm(alvo)
    alvo_clean = _area_clean(alvo_n)

    # 1) subáreas via Herança
    subareas = []
    if areaher_loc:
        for a in all_areas:
            if a == alvo:
                continue
            if _is_parent_of(alvo_n, a, areaher_loc):
                subareas.append(a)

    # 2) complemento via prefixo "limpo"
    for a in all_areas:
        if a == alvo:
            continue
        ac = _area_clean(a)
        if ac.startswith(alvo_clean) and len(ac) > len(alvo_clean):
            if a not in subareas:
                subareas.append(a)

    def _akey(a):
        return (len(_area_clean(a)), _area_clean(a))

    linhas = [alvo] + sorted(set(subareas), key=_akey)

    if isinstance(piv_index, (list, pd.Index)):
        return [a for a in linhas if a in piv_index]
    return linhas


def _rollup_full_counts(full_counts: dict, areaher: dict) -> dict:
    """
    full_counts = {area_full: n} com HC na área EXATA.

    Faz roll-up por Heranca:
      - cada área filha joga seu HC para todos os pais/avós.
    """
    out = dict(full_counts)
    for a_full, n in full_counts.items():
        a_full_n = _norm(a_full)
        for p in _area_parents(a_full_n, areaher):
            out[p] = out.get(p, 0) + n
    return out


def tabela_headcount_turnover(
    departamentos: pd.DataFrame,
    afastados: pd.DataFrame,
    movimentacoes: pd.DataFrame,
    anos=(2023, 2024, 2025),
    flatten=False
) -> pd.DataFrame:
    # 1) Base consolidada (universo de pessoas)
    base = _consolida_bases(departamentos, afastados)

    valid_ids = base["Cadastro"].astype(str).unique().tolist()

    # 2) Mapa área -> Herança
    areaher = _map_area_to_heranca(departamentos, afastados)

    # filtrar movimentações para só quem está na base
    mv = movimentacoes.copy()
    if not mv.empty:
        idcol_mv = _mov_id_col(mv)
        mv[idcol_mv] = mv[idcol_mv].astype(str)
        mv = mv[mv[idcol_mv].isin(valid_ids)]
    else:
        mv = movimentacoes

    # 3) HC, entradas externas, saídas externas e transferências internas
    hc_year = _headcount_por_ano(base, mv, anos, areaher)

    # >>> REATIVAR ÂNCORA DO ÚLTIMO ANO AQUI <<<
    if anos:
        ultimo_ano = max(anos)
        dt_fim_ano = pd.Timestamp(ultimo_ano, 12, 31)
        hc_snap = _hc_snapshot_from_departamentos(departamentos, areaher)
        if isinstance(hc_snap, pd.Series) and not hc_snap.empty:
            hc_year[ultimo_ano] = hc_snap

    # usa as movimentações já filtradas
    e_ext   = _entradas_por_ano(afastados, departamentos, anos, areaher, mv)

    # restringe saídas ao mesmo universo da base
    s_ext   = _saidas_por_ano(afastados, anos, areaher, valid_ids=valid_ids)

    # usa as movimentações já filtradas
    e_trf, s_trf = _transfer_por_ano(mv, anos, areaher)

    # conjuntos totais por ano (externas + transferências)
    e_sets_by_year = {}
    s_sets_by_year = {}
    for y in anos:
        e_sets_by_year[y] = _combine_event_sets(e_ext.get(y, {}), e_trf.get(y, {}))
        s_sets_by_year[y] = _combine_event_sets(s_ext.get(y, {}), s_trf.get(y, {}))

    # 4) Monta tabela de áreas x Herança (igual antes)
    dep_h = _rename_heranca(departamentos).copy()
    afa_h = _rename_heranca(afastados).copy()
    for df in (dep_h, afa_h):
        if "Area" in df.columns:
            df["Area"] = df["Area"].map(_norm)
        if "Heranca" in df.columns:
            df["Heranca"] = df["Heranca"].map(_norm)

    # NOVO: captura todas as áreas que aparecem em departamentos/afastados,
    # mesmo que não tenham Heranca ou eventos de HC/E/S nos anos analisados.
    areas_dep_afast = set()
    if "Area" in dep_h.columns:
        areas_dep_afast |= set(
            dep_h["Area"].dropna().astype(str).map(_norm)
        )
    if "Area" in afa_h.columns:
        areas_dep_afast |= set(
            afa_h["Area"].dropna().astype(str).map(_norm)
        )

    frames = []
    if {"Area", "Heranca"}.issubset(dep_h.columns):
        tmp = dep_h[["Area", "Heranca"]].copy()
        tmp["Departamento"] = dep_h.get("Departamento", "")
        frames.append(tmp)

    if {"Area", "Heranca"}.issubset(afa_h.columns):
        tmp = afa_h[["Area", "Heranca"]].copy()
        tmp["Departamento"] = afa_h.get("Departamento", "")
        frames.append(tmp)

    if frames:
        her_df = (
            pd.concat(frames, ignore_index=True)
              .dropna(subset=["Area"])
              .drop_duplicates(subset=["Area"])
              .sort_values("Area")
        )
    else:
        her_df = pd.DataFrame(columns=["Area", "Departamento", "Heranca"])

    # 5) Descobre todas as áreas que aparecem em HC/E/S
    areas_eventos = set()
    for y in anos:
        areas_eventos |= set(hc_year.get(y, pd.Series(dtype=int)).index)
        for d in (e_ext.get(y, {}), s_ext.get(y, {}), e_trf.get(y, {}), s_trf.get(y, {})):
            areas_eventos |= set(d.keys())

    # catálogo vindo da tabela de hierarquia (quando existir)
    areas_catalogo = set(her_df["Area"].astype(str)) if not her_df.empty else set()
    # NOVO: força inclusão de qualquer área que apareça em departamentos/afastados,
    # mesmo que não tenha aparecido em HC/E/S nos anos filtrados.
    areas_catalogo |= areas_dep_afast

    # extra: gptw, se existir
    extra_gptw_areas = set()
    df_gptw = globals().get("df_gptw", None)
    if isinstance(df_gptw, pd.DataFrame) and not df_gptw.empty:
        area_col_g = "Área" if "Área" in df_gptw.columns else ("Area" if "Area" in df_gptw.columns else None)
        if area_col_g is not None:
            extra_gptw_areas = set(
                df_gptw[area_col_g].dropna().astype(str).map(_norm).unique()
            )

    all_areas = sorted(areas_catalogo | areas_eventos | extra_gptw_areas)

    # 6) Ordenação hierárquica de áreas (igual antes)
    ordered_areas = []
    usados = set()
    macros = sorted({a.split("-", 1)[0] for a in all_areas})

    for macro in macros:
        bloco = _areas_macro_e_subareas(macro, all_areas, her_df=her_df)
        for a in bloco:
            if a in all_areas and a not in usados:
                ordered_areas.append(a)
                usados.add(a)

    for a in all_areas:
        if a not in usados:
            ordered_areas.append(a)

    areas = ordered_areas

    # snapshot atual de departamentos por área (ancoragem para as áreas vivas)
    dep_snapshot = _hc_snapshot_from_departamentos(departamentos, areaher)

    # retropropaga HC dos anos anteriores usando os fluxos E/S:
    # - áreas com gente em departamentos: recálculo backwards a partir da âncora;
    # - áreas sem ninguém em departamentos: 2025 = 0, anos anteriores = HC direto (afastados + dep).
    hc_year = _recalc_hc_backwards(
        hc_year,
        e_sets_by_year,
        s_sets_by_year,
        anos,
        areas,
        dep_snapshot=dep_snapshot,
    )

    # 7) Monta as colunas de HC, E, S por ano
    cols, data = [], {}
    for y in anos:
        hc = hc_year.get(y, pd.Series(dtype=int)).reindex(areas).fillna(0).astype(int)

        e_sets = e_sets_by_year.get(y, {})
        s_sets = s_sets_by_year.get(y, {})

        e_cnt = _sets_to_series(e_sets, areas)
        s_cnt = _sets_to_series(s_sets, areas)

        data[(y, "headcount")] = hc.values
        data[(y, "e")] = e_cnt.values
        data[(y, "s")] = s_cnt.values
        cols.extend([(y, "headcount"), (y, "e"), (y, "s")])

    out = pd.DataFrame(data, index=areas)
    out.index.name = "Área"
    out.columns = pd.MultiIndex.from_tuples(cols, names=["Ano", "Métrica"])
    out = out.reset_index()

    # 8) Anexa Herança
    if not her_df.empty:
        mapa_her_df = dict(zip(her_df["Area"], her_df["Heranca"]))
        her_final = out["Área"].map(mapa_her_df).fillna("")
    else:
        her_final = ""
    out.insert(1, "Heranca", her_final)

    # 9) Flatten opcional (igual antes)
    if flatten:
        sufixos = {"headcount": "hc", "e": "e", "s": "s"}
        flat = out.copy()
        col_tuples = flat.columns.tolist()
        novas = ["Área", "Heranca"] + [
            f"{a} {sufixos[m]}" for (a, m) in col_tuples[2:]
        ]
        flat.columns = novas
        return flat

    return out


def validar_departamentos_eventos(
    departamentos: pd.DataFrame,
    afastados: pd.DataFrame,
    movimentacoes: pd.DataFrame,
):
    """
    Verifica se alguma pessoa que está em 'departamentos' já teve:

      1) Demissão em 'afastados' (Data_Dem preenchida)
      2) Movimentação para outra área em 'movimentacoes' cuja
         última 'Abreviação UO' seja diferente da 'Area' atual.
    """
    dep = _rename_heranca(departamentos).copy()
    dep["Cadastro"] = dep["Cadastro"].astype(str)

    afa = _rename_heranca(afastados).copy()
    if "Cadastro" in afa.columns:
        afa["Cadastro"] = afa["Cadastro"].astype(str)
        afa["Data_Adm"] = _to_dt(afa["Data_Adm"])
        afa["Data_Dem"] = _to_dt(afa["Data_Dem"])

    mv = movimentacoes.copy()
    idcol = _mov_id_col(mv)
    mv[idcol] = mv[idcol].astype(str)
    mv["Data de alteração"] = _to_dt(mv["Data de alteração"])

    # 1) Em departamentos mas com demissão registrada
    if {"Cadastro", "Area", "Heranca", "Data_Adm", "Data_Dem"}.issubset(afa.columns):
        dep_afa = dep.merge(
            afa[["Cadastro", "Area", "Heranca", "Data_Adm", "Data_Dem"]],
            on="Cadastro",
            how="left",
            suffixes=("_dep", "_afa"),
        )
        dep_com_demissao = dep_afa[dep_afa["Data_Dem"].notna()].copy()
    else:
        dep_com_demissao = pd.DataFrame()

    # 2) Em departamentos com última mov. em outra área
    if {"Abreviação UO", "Abreviação UO antiga", "Data de alteração"}.issubset(mv.columns):
        mv_valid = mv[mv["Data de alteração"].notna()].sort_values(
            [idcol, "Data de alteração"]
        )
        last_mv = mv_valid.groupby(idcol).tail(1)[
            [idcol, "Abreviação UO", "Abreviação UO antiga", "Data de alteração"]
        ].copy()
        last_mv = last_mv.rename(columns={idcol: "Cadastro"})
        last_mv["Abreviação UO"] = last_mv["Abreviação UO"].map(_norm)

        dep2 = dep.copy()
        dep2["Area_norm"] = dep2["Area"].map(_norm)

        dep_mv = dep2.merge(last_mv, on="Cadastro", how="left")

        cond_diff_area = (
            dep_mv["Abreviação UO"].notna()
            & (dep_mv["Abreviação UO"] != dep_mv["Area_norm"])
        )
        dep_com_mov_outra_area = dep_mv[cond_diff_area].copy()
    else:
        dep_com_mov_outra_area = pd.DataFrame()

    pids_prob = set(dep_com_demissao.get("Cadastro", [])) | set(
        dep_com_mov_outra_area.get("Cadastro", [])
    )
    if not pids_prob:
        dep_com_demissao_ou_mov = pd.DataFrame(columns=dep.columns)
    else:
        dep_com_demissao_ou_mov = dep[dep["Cadastro"].isin(pids_prob)].copy()

    return {
        "dep_com_demissao": dep_com_demissao,
        "dep_com_mov_outra_area": dep_com_mov_outra_area,
        "dep_com_demissao_ou_mov": dep_com_demissao_ou_mov,
    }



#### Criando o dataframe 'headcount':

In [ ]:
# Inserindo o resultado da função dentro do df:
headcount = tabela_headcount_turnover(departamentos, afastados, movimentacoes, flatten=True)

#### Utilizando apenas as colunas necessárias para desenvolver os dashs:

In [ ]:
# df_semana_merged
df_semana_merged = df_semana_merged[[
    'Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca',
    'Cargo Especifico', 'Cargo', 'Afastado', 'Ano', 'Semana por Ano',
    'Contagem Presença'
]]

# df_datas
df_datas = df_datas[[
    'Cadastro', 'Planta', 'Data', 'Dia Semana', 'Semana por Ano', 'Ano'
]]

# df_almoco  (mantém Data e colunas possivelmente úteis)
df_almoco = df_almoco[[
    'Cadastro', 'Nome', 'Data', 'Hora', 'Dir', 'Planta', 'Coletor',
    'Descricao Coletor', 'Cargo', 'Afastado', 'Departamento', 'Heranca',
    'Cargo Especifico'
]]

# df_departamentos
departamentos = departamentos[[
    'Cadastro', 'Departamento', 'Area', 'Data_Admissão'
]]

# df_afastados
afastados = afastados[[
    'Cadastro', 'Departamento', 'Area', 'Data_Adm', 'Data_Dem'
]]

# df_gptw
gptw = gptw[[
    'Área', '2025',
    '2024', '2023'
]]

# 3. Carregando os Dados:

In [ ]:
# Exportando os Dados de presença na fábrica ou em casa:
df_semana_merged.to_parquet(R"..\project\database\ArquivoFinalSemana.parquet")
print("✅Carregado: '../project/database/ArquivoFinalSemana.parquet'")
df_datas.to_parquet(R"..\project\database\ArquivoFinalDatas.parquet")
print("✅Carregado: '../project/database/ArquivoFinalDatas.parquet'")
df_almoco.to_parquet(R"..\project\database\Almoco.parquet")
print("✅Carregado: '../project/database/Almoco.parquet'")

# Exportando os Dados mais característicos de cada pessoa:
departamentos.to_parquet(R"..\project\database\Departamentos.parquet")
print("✅Carregado: '../project/database/Departamentos.parquet'")
afastados.to_parquet(R"..\project\database\Afastados.parquet")
print("✅Carregado: '../project/database/Afastados.parquet'")
gptw.to_parquet(R"..\project\database\GPTW.parquet")
print("✅Carregado: '../project/database/GPTW.parquet'")
movimentacoes.to_parquet(R"..\project\database\Movimentacoes.parquet")
print("✅Carregado: '../project/database/Movimentacoes.parquet'")

# Exportando os Dados mais característicos de cada área:
headcount.to_parquet(R"..\project\database\HeadCount.parquet")
print("✅Carregado: '../project/database/HeadCount.parquet'")